In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

This document is a based on logistic_regression. It is a cleaned-up version. In this document we train logistic regression on one complication. In general we observe results close to those of multiple regression.

## Data

In [ ]:
path_train = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
path_test = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\processed\A\global_test_set_non-iid.csv"

df = pd.read_csv(path_train)
test_data = pd.read_csv(path_test)

"""
df = (
    df.groupby("Client", group_keys=False)
      .apply(lambda x: x.sample(frac=20000 / len(df), random_state=42))
      .reset_index(drop=True)
)
"""


variables = df.columns[2:27].tolist()

# Ændre Target_col for øvrige complicationer.

target_col = "Risk_AlveolarOsteitis"
category_col = "Risk_Category_AlveolarOsteitis"
client_col = "Client"


scaler = StandardScaler()

train_design = pd.DataFrame(
    scaler.fit_transform(df[variables]),
    columns=variables,
    index=df.index
)

test_design = pd.DataFrame(
    scaler.transform(test_data[variables]),
    columns=variables,
    index=test_data.index
)

train_design[target_col] = df[target_col].values
train_design[category_col] = df[category_col].values
train_design[client_col] = df[client_col].values

test_design[target_col] = test_data[target_col].values
test_design[category_col] = test_data[category_col].values



In [31]:
len(df)

47000

## Functions

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def assign_tertiles(y_hat):
    q1 = np.quantile(y_hat, 1/3)
    q2 = np.quantile(y_hat, 2/3)

    groups = np.where(
        y_hat <= q1, 0,
        np.where(y_hat <= q2, 1, 2)
    )

    return groups, q1, q2


def metrics_for_one_complication(y_true, y_pred):
    results = {}

    for cls in [0, 1, 2]:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0
        )

        results[cls] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    macro_f1 = np.mean([results[cls]["f1"] for cls in [0, 1, 2]])

    return {
        "per_class": results,
        "macro_f1": macro_f1
    }


def federated_logistic_regression_binary(
    train_df,
    test_df,
    variables,
    target_col,
    category_col,
    client_col="Client",
    rounds=50,
    local_epochs=5,
    lr=0.01,
    weighted=True,
    random_state=42
):
    n_features = len(variables)

    global_coef = np.zeros((1, n_features))
    global_intercept = np.zeros(1)

    X_test = test_df[variables].to_numpy(dtype=float)
    y_true_group = test_df[category_col].to_numpy(dtype=int)

    history = []

    for rnd in range(rounds):

        old_coef = global_coef.copy()
        old_intercept = global_intercept.copy()

        local_coefs = []
        local_intercepts = []
        local_sizes = []

        for client_id in sorted(train_df[client_col].unique()):

            client_df = train_df[train_df[client_col] == client_id]

            X_client = client_df[variables].to_numpy(dtype=float)
            y_client = client_df[target_col].to_numpy(dtype=int)

            model = SGDClassifier(
                loss="log_loss",
                penalty=None,
                learning_rate="constant",
                eta0=lr,
                max_iter=1,
                tol=None,
                warm_start=True,
                random_state=random_state
            )

            # Dummy initialisering
            model.partial_fit(
                X_client[:1],
                y_client[:1],
                classes=np.array([0, 1])
            )

            # Overskriv med globale vægte
            model.coef_ = global_coef.copy()
            model.intercept_ = global_intercept.copy()
            model.classes_ = np.array([0, 1])

            # Lokal træning
            for _ in range(local_epochs):
                model.partial_fit(X_client, y_client)

            local_coefs.append(model.coef_.copy())
            local_intercepts.append(model.intercept_.copy())
            local_sizes.append(len(client_df))

        local_sizes = np.array(local_sizes, dtype=float)

        if weighted:
            global_coef = np.average(local_coefs, axis=0, weights=local_sizes)
            global_intercept = np.average(local_intercepts, axis=0, weights=local_sizes)
        else:
            global_coef = np.mean(local_coefs, axis=0)
            global_intercept = np.mean(local_intercepts, axis=0)

        coef_change = np.linalg.norm(global_coef - old_coef)
        intercept_change = np.linalg.norm(global_intercept - old_intercept)

        logits = X_test @ global_coef.T + global_intercept
        probs = sigmoid(logits).ravel()

        y_pred_group, q1, q2 = assign_tertiles(probs)

        metrics = metrics_for_one_complication(y_true_group, y_pred_group)
        macro_f1 = metrics["macro_f1"]

        history.append({
            "round": rnd + 1,
            "macro_f1": macro_f1,
            "q1": q1,
            "q2": q2,
            "coef_change": coef_change,
            "intercept_change": intercept_change
        })

        print(
            f"Round {rnd+1:03d} | Macro F1 = {macro_f1:.4f} | "
            f"coef_change = {coef_change:.6f} | "
            f"intercept_change = {intercept_change:.6f}"
        )

    return global_intercept, global_coef, pd.DataFrame(history)

## Traning

In [29]:
intercept, coefs, history_df = federated_logistic_regression_binary(
    train_df=train_design,
    test_df=test_design,
    variables=variables,
    target_col=target_col,
    category_col=category_col,
    client_col=client_col,
    rounds=50,
    local_epochs=1,
    lr=0.001,
    weighted=True
)

Round 001 | Macro F1 = 0.6241 | coef_change = 0.156514 | intercept_change = 1.311501
Round 002 | Macro F1 = 0.6408 | coef_change = 0.109957 | intercept_change = 0.503203
Round 003 | Macro F1 = 0.6555 | coef_change = 0.090781 | intercept_change = 0.249536
Round 004 | Macro F1 = 0.6632 | coef_change = 0.073441 | intercept_change = 0.142320
Round 005 | Macro F1 = 0.6702 | coef_change = 0.058318 | intercept_change = 0.088731
Round 006 | Macro F1 = 0.6726 | coef_change = 0.045881 | intercept_change = 0.058925
Round 007 | Macro F1 = 0.6766 | coef_change = 0.035988 | intercept_change = 0.040993
Round 008 | Macro F1 = 0.6779 | coef_change = 0.028240 | intercept_change = 0.029517
Round 009 | Macro F1 = 0.6779 | coef_change = 0.022206 | intercept_change = 0.021803
Round 010 | Macro F1 = 0.6766 | coef_change = 0.017506 | intercept_change = 0.016413
Round 011 | Macro F1 = 0.6766 | coef_change = 0.013836 | intercept_change = 0.012531
Round 012 | Macro F1 = 0.6772 | coef_change = 0.010959 | intercep

## Results

In [30]:
X_test_final = test_design[variables].to_numpy(dtype=float)
y_true_group = test_design[category_col].to_numpy(dtype=int)

logits = X_test_final @ coefs.T + intercept
probs = sigmoid(logits).ravel()

y_pred_group, q1, q2 = assign_tertiles(probs)

final_metrics = metrics_for_one_complication(y_true_group, y_pred_group)

print("\nFinal macro F1:")
print(final_metrics["macro_f1"])

print("\nPer class metrics:")
for cls, vals in final_metrics["per_class"].items():
    print(f"\nClass {cls}")
    print(vals)

print("\nTertile cutoffs:")
print("q1:", q1)
print("q2:", q2)

history_df


Final macro F1:
0.6732468332319401

Per class metrics:

Class 0
{'TP': np.int64(771), 'FP': np.int64(229), 'FN': np.int64(194), 'precision': np.float64(0.771), 'recall': np.float64(0.7989637305699482), 'f1': np.float64(0.7847328244274809)}

Class 1
{'TP': np.int64(539), 'FP': np.int64(461), 'FN': np.int64(488), 'precision': np.float64(0.539), 'recall': np.float64(0.5248296007789679), 'f1': np.float64(0.5318204242723236)}

Class 2
{'TP': np.int64(706), 'FP': np.int64(294), 'FN': np.int64(302), 'precision': np.float64(0.706), 'recall': np.float64(0.7003968253968254), 'f1': np.float64(0.7031872509960159)}

Tertile cutoffs:
q1: 0.05360461832804041
q2: 0.10879681806285


,round,macro_f1,q1,q2,coef_change,intercept_change
0,1,0.624081,0.199679,0.224850,0.156514,1.311501
1,2,0.640838,0.125561,0.156928,0.109957,0.503203
2,3,0.655523,0.096809,0.132940,0.090781,0.249536
3,4,0.663215,0.082493,0.122024,0.073441,0.142320
4,5,0.670234,0.074502,0.116874,0.058318,0.088731
5,6,0.672592,0.068934,0.113881,0.045881,0.058925
6,7,0.676587,0.065270,0.112087,0.035988,0.040993
7,8,0.677908,0.062754,0.110871,0.028240,0.029517
8,9,0.677898,0.060736,0.110104,0.022206,0.021803
9,10,0.676562,0.059415,0.109583,0.017506,0.016413


Bemærk i den multiple regressions model får vi 0.675. Resultaterne er altså ens (+- støj). Multipel regression er mere simpel.